# Model Monitoring: Drift Detection and Performance Tracking

This notebook demonstrates how to monitor registered models in the Snowflake
Model Registry using `snowflakeR`. Model monitoring (ML Observability) tracks
prediction drift, model performance, and feature statistics over time.

**What you'll learn:**
- Setting up a model monitor with source data and configuration
- Querying performance, drift, and statistics metrics from R
- Managing monitor lifecycle (suspend, resume, segments)
- Integrating with `vetiver` for R-native metric visualization

**Prerequisites:** A registered model with a `task` type (e.g., `TABULAR_REGRESSION`).
This notebook creates its own test model and prediction data.

**Sections:**
1. Setup
2. Connect & Registry
3. Register a Model for Monitoring
4. Create a Monitor
5. List and Get Monitors
6. Query Metrics
7. Monitor Lifecycle
8. vetiver Integration
9. Cleanup


In [ ]:
import time as _time
_nb_start = _time.time()
print(f"Notebook started: {_time.strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Setup

Single cell to set session context, validate EAI, install R, and install packages.


In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_config.yaml", packages=["snowflakeR"])


## 2. Connect & Registry with Monitoring Enabled

Model monitoring requires `enable_monitoring = TRUE` in the registry options.


In [ ]:
%%R
library(snowflakeR)

conn <- sfr_connect()
conn

# Create registry with monitoring enabled
reg <- sfr_model_registry(conn, options = list(enable_monitoring = TRUE))
reg


---

## 3. Register a Model for Monitoring

Monitoring requires a model logged with a **task** type (e.g., `TABULAR_REGRESSION`,
`TABULAR_BINARY_CLASSIFICATION`). We'll train a simple model and create a
prediction source table that simulates logged inference results.


In [ ]:
%%R
# Train and register a model with task type
model <- lm(mpg ~ wt + hp, data = mtcars)

mv <- sfr_log_model(
  reg,
  model        = model,
  model_name   = "SFR_DEMO_MONITORED",
  version_name = "V1",
  input_cols   = list(wt = "double", hp = "double"),
  output_cols  = list(prediction = "double"),
  comment      = "Linear regression for monitoring demo",
  task         = "TABULAR_REGRESSION"
)

mv


In [ ]:
%%R
# Create a prediction source table (simulating logged inference results)
# This table must have: timestamp, predictions, actuals, and optional IDs
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", sfr_fqn(conn, "SFR_DEMO_PREDICTIONS"), "(",
  "  ID         VARCHAR,",
  "  WT         FLOAT,",
  "  HP         FLOAT,",
  "  PREDICTION FLOAT,",
  "  ACTUAL_MPG FLOAT,",
  "  EVENT_TIME TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()",
  ")"
))

# Insert predictions with known actuals
sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_PREDICTIONS"),
  "(ID, WT, HP, PREDICTION, ACTUAL_MPG) VALUES",
  "('r1', 2.620, 110, 22.6, 21.0),",
  "('r2', 3.215, 110, 20.9, 21.4),",
  "('r3', 3.440, 175, 16.5, 18.1),",
  "('r4', 3.460, 245, 13.2, 14.3),",
  "('r5', 2.200, 93,  27.1, 30.4),",
  "('r6', 3.570, 245, 12.8, 15.2),",
  "('r7', 2.780, 175, 19.2, 19.2),",
  "('r8', 3.150, 150, 18.4, 17.8)"
))

rcat("Prediction source table created with 8 rows.")


---

## 4. Create a Monitor

A monitor consists of two parts:
- **Source config:** Points to the table with predictions and actuals
- **Monitor config:** Links to the model version, sets the warehouse and aggregation window


In [ ]:
%%R
# Define the prediction source
src <- sfr_monitor_source(
  source                   = sfr_fqn(conn, "SFR_DEMO_PREDICTIONS"),
  timestamp_column         = "EVENT_TIME",
  prediction_score_columns = "PREDICTION",
  actual_score_columns     = "ACTUAL_MPG",
  id_columns               = "ID"
)

rcat("Monitor source:")
src


In [ ]:
%%R
# Define the monitor configuration
cfg <- sfr_monitor_config(
  model_name         = "SFR_DEMO_MONITORED",
  version_name       = "V1",
  warehouse          = conn$warehouse,
  aggregation_window = "1 day"
)

rcat("Monitor config:")
cfg


In [ ]:
%%R
# Create the monitor
sfr_add_monitor(reg, "SFR_DEMO_MONITOR", src, cfg)
rcat("Monitor created!")


In [ ]:
%%R
# Inspect monitor details (SQL-direct)
desc <- sfr_describe_monitor(conn, "SFR_DEMO_MONITOR")
desc


---

## 5. List and Get Monitors


In [ ]:
%%R
# List all monitors in the registry
monitors <- sfr_show_model_monitors(reg)
rcat(sprintf("Total monitors: %d", nrow(monitors)))
monitors


In [ ]:
%%R
# Get a specific monitor by name
mon <- sfr_get_monitor(reg, name = "SFR_DEMO_MONITOR")
mon


In [ ]:
%%R
# Get by model/version (alternative)
mon2 <- sfr_get_monitor(reg, model_name = "SFR_DEMO_MONITORED", version_name = "V1")
mon2


---

## 6. Query Metrics (SQL-direct)

Monitor metrics are queried via SQL table functions. For convenience,
`snowflakeR` wraps these as R functions that return plain `data.frame`
objects -- ready for `dplyr`, `ggplot2`, or any R analysis tool.

Three metric types:
- **Performance:** RMSE, MAE, R-squared, accuracy, precision, recall, etc.
- **Drift:** Jensen-Shannon, KL divergence, etc. (requires a baseline)
- **Stats:** COUNT, COUNT_NULL, MIN, MAX, AVG, SUM on individual columns


In [ ]:
%%R
# Query performance metrics (RMSE for regression)
perf <- sfr_monitor_performance(
  conn, "SFR_DEMO_MONITOR", "RMSE",
  granularity = "ALL",
  start_time  = Sys.Date() - 7,
  end_time    = Sys.Date() + 1
)

rcat("Performance metrics (RMSE):")
perf


In [ ]:
%%R
# Query feature statistics
stats <- sfr_monitor_stats(
  conn, "SFR_DEMO_MONITOR", "AVG", "PREDICTION",
  granularity = "ALL",
  start_time  = Sys.Date() - 7,
  end_time    = Sys.Date() + 1
)

rcat("Feature statistics (AVG of PREDICTION):")
stats


### Drift detection

Drift metrics compare current predictions against a **baseline** distribution.
A baseline must be configured on the monitor for drift queries to work.
Without a baseline, drift queries return an error.

```r
# Example (requires baseline):
drift <- sfr_monitor_drift(
  conn, "MY_MONITOR", "JENSEN_SHANNON", "PREDICTION",
  granularity = "DAY",
  start_time  = Sys.Date() - 30,
  end_time    = Sys.Date()
)
```


---

## 7. Monitor Lifecycle

Monitors can be suspended and resumed. You can also add segment columns
to slice metrics by category (e.g., by region, product type).


In [ ]:
%%R
# Suspend the monitor (pauses metric aggregation)
sfr_suspend_monitor(conn, "SFR_DEMO_MONITOR")
rcat("Monitor suspended.")


In [ ]:
%%R
# Resume the monitor
sfr_resume_monitor(conn, "SFR_DEMO_MONITOR")
rcat("Monitor resumed.")


### Segments

Segments let you slice metrics by a column in the source table. For example,
if your predictions table has a `REGION` column, you can add it as a segment
and query metrics per region.

```r
# Add a segment column
sfr_add_monitor_segment(conn, "MY_MONITOR", "REGION")

# Query metrics for a specific segment
sfr_monitor_performance(
  conn, "MY_MONITOR", "RMSE",
  granularity = "DAY",
  start_time  = Sys.Date() - 30,
  end_time    = Sys.Date(),
  segment     = list(column = "REGION", value = "US")
)

# Remove a segment
sfr_drop_monitor_segment(conn, "MY_MONITOR", "REGION")
```


---

## 8. vetiver Integration

`snowflakeR` bridges Snowflake monitoring with the R `vetiver` package for
model management:

- **Pull:** `sfr_monitor_to_vetiver()` converts Snowflake metrics to vetiver-format
  tibbles with `.index`, `.metric`, `.estimate` columns
- **Push:** `sfr_vetiver_to_metrics()` stores R-computed vetiver metrics as
  registry model metrics


In [ ]:
%%R
# Pull Snowflake metrics as vetiver-compatible tibble
vet_metrics <- sfr_monitor_to_vetiver(
  conn, "SFR_DEMO_MONITOR", "RMSE",
  metric_type = "performance",
  start_time  = Sys.Date() - 7,
  end_time    = Sys.Date() + 1
)

rcat("Vetiver-format metrics:")
vet_metrics


In [ ]:
%%R
# Visualize with vetiver (if installed)
if (requireNamespace("vetiver", quietly = TRUE)) {
  library(vetiver)
  vetiver_plot_metrics(vet_metrics)
} else {
  rcat("Install vetiver for metric plots: install.packages('vetiver')")
  rcat("The tibble above can also be used with ggplot2 directly.")
}


In [ ]:
%%R
# Push R-computed metrics to the registry
if (requireNamespace("vetiver", quietly = TRUE)) {
  library(vetiver)

  # Compute metrics locally
  pred_data <- sfr_query(conn, paste(
    "SELECT PREDICTION, ACTUAL_MPG FROM",
    sfr_fqn(conn, "SFR_DEMO_PREDICTIONS")
  ))

  local_metrics <- vetiver_compute_metrics(
    pred_data,
    truth    = "ACTUAL_MPG",
    estimate = "PREDICTION",
    metric_set = yardstick::metric_set(
      yardstick::rmse, yardstick::mae, yardstick::rsq
    )
  )

  # Store in registry
  sfr_vetiver_to_metrics(reg, "SFR_DEMO_MONITORED", "V1", local_metrics)
  rcat("Vetiver metrics stored in registry.")
} else {
  rcat("Skipping vetiver push (vetiver not installed).")
}


---

## 9. Cleanup


In [ ]:
%%R
# Uncomment to clean up demo objects
#
# sfr_delete_monitor(reg, "SFR_DEMO_MONITOR")
# sfr_delete_model(reg, "SFR_DEMO_MONITORED")
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_PREDICTIONS")))
# sfr_disconnect(conn)
# rcat("Cleanup complete.")


---

## Next steps

- **Model Consumption:** See `workspace_model_consumption.ipynb` for SQL-direct
  batch inference and cross-language model usage.
- **Feature Store:** See `workspace_feature_store.ipynb` for feature engineering
  with time aggregation and online serving.
- **Experiment Tracking:** See `workspace_model_registry.ipynb` Section 8 for
  tracking hyperparameter runs with tidymodels integration.


In [ ]:
_nb_elapsed = _time.time() - _nb_start
_mins, _secs = divmod(_nb_elapsed, 60)
print(f"Notebook completed: {_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {int(_mins)}m {_secs:.1f}s")


In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MODEL_MONITORING_RESULTS AS
        SELECT 'model_monitoring' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MODEL_MONITORING_RESULTS")
except Exception:
    pass
